# **1. Limpeza dos dados**

Este notebook une as duas fontes de partidas numa única tabela, com uma linha por partida da Série A. O calendário da CBF mantém o horário e o estádio oficiais; a ESPN contribui com o público e um nome curto para cada clube. A limpeza separa três situações que um modelo de previsão não deve misturar: jogos regulares com torcida, jogos sem o público registrado e jogos disputados com restrição de público.

| etapa | fonte | resultado |
|---|---|---|
| horário | data e hora da CBF, horário UTC da ESPN | data e hora locais de cada partida |
| identidade dos clubes | semelhança dos nomes na mesma data | um identificador da ESPN para cada clube da CBF |
| junção das partidas | data, mandante e visitante | público associado à linha da CBF |
| situação do público | público em relação ao nível do clube | regular, ausente ou com restrição |
| estádio | campo de local da CBF | estádio, cidade e estado padronizados |

In [1]:
from __future__ import annotations

import unicodedata
from datetime import timedelta
from difflib import SequenceMatcher
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

### **1.1. Configuração**

`RESTRICTION_SHARE` define um jogo com restrição: público abaixo dessa fração da mediana do clube em casa na mesma temporada, no estádio principal do clube naquela temporada. Portões fechados por punição e venda para um único setor ficam abaixo desse limite; um público baixo em outro estádio é efeito do estádio e continua como jogo regular.

In [2]:
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

INTERIM = ROOT / "data" / "interim"
PROCESSED = ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

ATTENDANCE_SEASONS = list(range(2022, 2027))
MIN_VALID_ATTENDANCE = 200
RESTRICTION_SHARE = 0.20
LOCAL_TZ = "America/Sao_Paulo"

cbf = pd.read_csv(INTERIM / "cbf_games.csv", parse_dates=["date"])
espn = pd.read_csv(INTERIM / "espn_events.csv")
print(f"partidas CBF: {len(cbf):,} | eventos ESPN: {len(espn):,}")

partidas CBF: 2,660 | eventos ESPN: 1,883


## **2. Horário da partida**

A CBF publica o horário no fuso de Brasília. A ESPN guarda o horário em UTC, então sua data é convertida para Brasília antes da junção; um jogo às 21h30 já está no dia seguinte em UTC.

In [3]:
espn["kickoff_local"] = pd.to_datetime(espn["kickoff_utc"], utc=True).dt.tz_convert(LOCAL_TZ).dt.tz_localize(None)
espn["date"] = espn["kickoff_local"].dt.normalize()

kickoff_time = pd.to_datetime(cbf["kickoff"], format="%H:%M", errors="coerce")
cbf["kickoff_local"] = cbf["date"] + pd.to_timedelta(kickoff_time.dt.hour.fillna(0), unit="h") + pd.to_timedelta(kickoff_time.dt.minute.fillna(0), unit="m")
cbf.loc[kickoff_time.isna() | cbf["date"].isna(), "kickoff_local"] = pd.NaT
print(f"partidas CBF sem horário: {cbf['kickoff_local'].isna().sum()}")

partidas CBF sem horário: 0


## **3. Identidade dos clubes**

As duas fontes escrevem os nomes de forma diferente (`Botafogo de Futebol E Regatas` contra `Botafogo`), e a CBF ainda cria um novo identificador quando o clube vira SAF (Atlético Mineiro, Vasco, Fortaleza, Coritiba e outros aparecem com dois identificadores). O identificador da ESPN é estável e por isso passa a ser a chave canônica de cada clube. Para cada partida da CBF depois da pandemia, os eventos da ESPN do mesmo dia são pontuados pela semelhança dos nomes dos dois clubes, e o melhor par vota no mapeamento de cada identificador da CBF.

In [4]:
ALIASES = {"atletico mineiro": "atletico mg", "athletico paranaense": "athletico pr", "sport recife": "sport", "america": "america mineiro", "atletico": "atletico goianiense"}


def fold(value) -> str:
    text = "".join(c for c in unicodedata.normalize("NFKD", str(value)) if not unicodedata.combining(c)).casefold()
    for noise in ("esporte clube", "futebol clube", "de futebol e regatas", " saf", " fc", " ec", "sport club ", "-"):
        text = text.replace(noise, " ")
    text = " ".join(text.split())
    return ALIASES.get(text, text)


def similarity(a: str, b: str) -> float:
    a, b = fold(a), fold(b)
    if a == b:
        return 1.0
    if a in b or b in a:
        return 0.9
    return SequenceMatcher(None, a, b).ratio()


post = cbf[cbf["season"].isin(ATTENDANCE_SEASONS) & cbf["date"].notna()]
espn_by_date = {day: frame for day, frame in espn.groupby("date")}
votes = []
for row in post.itertuples(index=False):
    candidates = pd.concat([espn_by_date.get(row.date + timedelta(days=k), espn.iloc[:0]) for k in (0, 1, -1)])
    if candidates.empty:
        continue
    scores = [similarity(row.home_team, c.espn_home) + similarity(row.away_team, c.espn_away) for c in candidates.itertuples(index=False)]
    best = candidates.iloc[int(np.argmax(scores))]
    if max(scores) >= 1.4:
        votes += [(row.home_id, best.espn_home_id, best.espn_home), (row.away_id, best.espn_away_id, best.espn_away)]

votes = pd.DataFrame(votes, columns=["cbf_id", "espn_id", "club"])
voted = votes.groupby(["cbf_id", "espn_id", "club"]).size().rename("votes").reset_index().sort_values("votes", ascending=False).drop_duplicates("cbf_id")
cbf_names = pd.concat([
    cbf[["home_id", "home_team"]].set_axis(["cbf_id", "cbf_name"], axis=1),
    cbf[["away_id", "away_team"]].set_axis(["cbf_id", "cbf_name"], axis=1),
]).drop_duplicates("cbf_id")
club_map = cbf_names.merge(voted, on="cbf_id", how="left")
club_map["method"] = np.where(club_map["espn_id"].notna(), "same-day vote", None)
display(club_map[club_map["espn_id"].notna()].sort_values("club"))

,cbf_id,cbf_name,espn_id,club,votes,method
26,59897,América,6154.0,América Mineiro,76.0,same-day vote
17,20052,Athletico Paranaense,3458.0,Athletico-PR,150.0,same-day vote
19,20093,Atlético,10357.0,Atlético Goianiense,38.0,same-day vote
33,62261,Atlético Goianiense Saf,10357.0,Atlético Goianiense,38.0,same-day vote
14,20003,Atlético Mineiro,7632.0,Atlético-MG,38.0,same-day vote
27,62194,Atlético Mineiro,7632.0,Atlético-MG,149.0,same-day vote
25,20058,Avaí,9966.0,Avaí,38.0,same-day vote
31,61377,Bahia,9967.0,Bahia,150.0,same-day vote
24,60175,Botafogo,6086.0,Botafogo,188.0,same-day vote
18,20031,Ceará,9969.0,Ceará,76.0,same-day vote


### **3.1. Clubes das temporadas da pandemia**

Identificadores da CBF usados apenas em 2020 e 2021 nunca encontram um evento da ESPN, porque o público é coletado a partir de 2022. Eles são mapeados pelo nome para os clubes da ESPN já identificados, de modo que o Elo de um clube continua desde 2020 mesmo quando o identificador da CBF mudou. Clubes sem correspondência recebem uma chave própria derivada do identificador da CBF.

In [5]:
espn_clubs = voted.drop_duplicates("espn_id")[["espn_id", "club"]]
for i, row in club_map[club_map["espn_id"].isna()].iterrows():
    scores = espn_clubs["club"].map(lambda name: similarity(row.cbf_name, name))
    if scores.max() >= 0.85:
        best = espn_clubs.loc[scores.idxmax()]
        club_map.loc[i, ["espn_id", "club", "method"]] = [best.espn_id, best.club, "name"]
club_map["club_id"] = club_map["espn_id"].fillna(100_000 + club_map["cbf_id"]).astype(int)
club_map["club"] = club_map["club"].fillna(club_map["cbf_name"])
club_map["method"] = club_map["method"].fillna("unmatched")
display(club_map[club_map["method"].ne("same-day vote")].sort_values("club"))
assert club_map["cbf_id"].is_unique, "Um identificador da CBF tem dois mapeamentos"
assert club_map.loc[club_map["cbf_id"].isin(post["home_id"]), "method"].eq("same-day vote").all(), "Um clube do período estudado não foi associado à ESPN"

,cbf_id,cbf_name,espn_id,club,votes,method,club_id
22,20047,America,6154.0,América Mineiro,NaN,name,6154
16,20006,Esporte Clube Bahia,9967.0,Bahia,NaN,name,9967
1,20004,Botafogo de Futebol E Regatas,6086.0,Botafogo,NaN,name,6086
11,20012,Vasco da Gama,3454.0,Vasco da Gama,NaN,name,3454


## **4. Junção das partidas**

Com os identificadores definidos, cada partida da CBF é unida ao evento da ESPN com o mesmo mandante e visitante, primeiro na mesma data local e depois com até dois dias de diferença. A partir daqui, `home_id` e `away_id` guardam a chave canônica do clube.

In [6]:
id_to_club = club_map.set_index("cbf_id")["club_id"]
id_to_name = club_map.set_index("cbf_id")["club"]

matches = cbf.rename(columns={"home_id": "cbf_home_id", "away_id": "cbf_away_id"})
matches["home_id"] = matches["cbf_home_id"].map(id_to_club)
matches["away_id"] = matches["cbf_away_id"].map(id_to_club)
matches["home"] = matches["cbf_home_id"].map(id_to_name)
matches["away"] = matches["cbf_away_id"].map(id_to_name)

espn_keyed = espn[["espn_id", "date", "espn_home_id", "espn_away_id", "attendance", "espn_venue", "status"]].rename(columns={"espn_home_id": "home_id", "espn_away_id": "away_id"})
joined = []
remaining = matches[matches["season"].isin(ATTENDANCE_SEASONS)]
for shift in (0, 1, -1, 2, -2):
    if remaining.empty:
        break
    attempt = remaining.assign(join_date=remaining["date"] + timedelta(days=shift)).merge(
        espn_keyed.rename(columns={"date": "join_date"}), on=["join_date", "home_id", "away_id"], how="inner"
    )
    joined.append(attempt.drop(columns="join_date").assign(date_shift=shift))
    remaining = remaining[~remaining["game_id"].isin(attempt["game_id"])]
joined = pd.concat(joined, ignore_index=True).drop_duplicates("game_id")

matches = matches.merge(joined[["game_id", "espn_id", "attendance", "espn_venue", "status", "date_shift"]], on="game_id", how="left")
post_played = matches["season"].isin(ATTENDANCE_SEASONS) & matches["home_goals"].notna() & matches["date"].lt(pd.Timestamp.today().normalize())
print(f"partidas disputadas depois da pandemia: {post_played.sum():,} | unidas à ESPN: {matches.loc[post_played, 'espn_id'].notna().sum():,}")
display(matches.loc[post_played & matches["espn_id"].isna(), ["season", "round", "date", "home", "away", "venue_raw"]].head(20))
display(joined["date_shift"].value_counts().to_frame("partidas"))

partidas disputadas depois da pandemia: 1,880 | unidas à ESPN: 1,880


,season,round,date,home,away,venue_raw


,partidas
date_shift,
0,1879
-1,1


## **5. Estádio**

O campo de local da CBF é limpo e separado em estádio, cidade e estado. Os nomes já são harmonizados pela CBF entre as temporadas (`Nubank Parque` cobre todos os jogos do Allianz Parque), então apenas espaços e acentos são normalizados na chave.

In [7]:
for column in ("stadium", "city", "state"):
    matches[column] = matches[column].astype("string").str.replace(r"\s+", " ", regex=True).str.strip()
matches["venue_key"] = matches["stadium"].map(fold, na_action="ignore") + " | " + matches["city"].map(fold, na_action="ignore")
display(matches.loc[post_played].groupby("stadium").size().sort_values(ascending=False).head(15).to_frame("partidas"))

,partidas
stadium,
Maracanã,184
Arena Castelão,110
Neo Química Arena,94
Nilton Santos,93
Mineirão,93
Beira-Rio,91
Morumbi,85
Nubank Parque,80
Arena da Baixada,75


## **6. Situação do público**

A ESPN registra zero ou um número simbólico quando o público é desconhecido, então valores abaixo de `MIN_VALID_ATTENDANCE` passam a ausentes. Isso afeta a maior parte de 2026: a ESPN deixou de publicar o público da Série A no fim de maio de 2026, e as partidas seguintes ficam sem público. Entre os valores válidos, um jogo no estádio principal do clube na temporada é marcado como restrito quando fica abaixo de `RESTRICTION_SHARE` da mediana do clube em casa naquela temporada. Jogos em outro estádio nunca são marcados: um estádio menor ou distante é exatamente o que as variáveis de estádio devem capturar. Os jogos restritos permanecem na tabela, mas ficam fora dos alvos de treino e do histórico, porque seu nível é definido por uma punição ou um fechamento, e não pela demanda.

In [8]:
matches["attendance"] = pd.to_numeric(matches["attendance"], errors="coerce")
matches.loc[matches["attendance"] < MIN_VALID_ATTENDANCE, "attendance"] = np.nan
season_median = matches.groupby(["home_id", "season"])["attendance"].transform("median")
main_venue = matches.loc[post_played].groupby(["home_id", "season"])["venue_key"].agg(lambda s: s.mode().iat[0])
main_stadium = pd.Series(list(zip(matches["home_id"], matches["season"])), index=matches.index).map(main_venue).eq(matches["venue_key"])
monthly_coverage = matches.loc[post_played].groupby(matches["date"].dt.to_period("M"))["attendance"].apply(lambda s: s.notna().mean())
display(monthly_coverage[monthly_coverage.index.year == 2026].round(2).to_frame("proporção com público, 2026"))
matches["crowd_status"] = np.select(
    [~post_played, matches["attendance"].isna(), main_stadium & (matches["attendance"] < RESTRICTION_SHARE * season_median)],
    ["not_studied", "missing", "restricted"],
    "regular",
)
display(matches.loc[post_played].groupby(["season", "crowd_status"]).size().unstack(fill_value=0))
display(matches.loc[matches["crowd_status"].eq("restricted"), ["season", "round", "date", "home", "away", "stadium", "attendance"]].assign(club_median=season_median).sort_values("date"))

,"proporção com público, 2026"
date,
2026-01,1.00
2026-02,1.00
2026-03,1.00
2026-04,1.00
2026-05,0.78
2026-07,0.21
2026-08,0.08
2026-09,0.03
2026-10,0.00


crowd_status,missing,regular,restricted
season,,,
2022,4,375,1
2023,12,367,1
2024,4,376,0
2025,0,380,0
2026,183,177,0


,season,round,date,home,away,stadium,attendance,club_median
813,2022,6,2022-05-14,Palmeiras,Red Bull Bragantino,Nubank Parque,2599.0,38888.0
1514,2023,38,2023-12-06,Goiás,América Mineiro,Hailé Pinheiro,1966.0,10260.0


## **7. Exportação e auditoria**

| arquivo | conteúdo |
|---|---|
| `data/processed/matches.csv` | uma linha por partida da Série A de 2020 a 2026, com horário, estádio, placar, público e situação do público |
| `data/processed/club_map.csv` | identificadores da CBF, chave canônica, identificador da ESPN e nome de cada clube |

A auditoria verifica a cobertura da junção, a unicidade da chave das partidas e a ausência de público antes de 2022.

In [9]:
columns = [
    "game_id", "espn_id", "season", "round", "date", "kickoff_local", "home_id", "home", "away_id", "away", "cbf_home_id", "cbf_away_id",
    "home_goals", "away_goals", "stadium", "city", "state", "venue_key", "attendance", "crowd_status",
]
matches = matches[columns].sort_values(["kickoff_local", "game_id"])
matches.to_csv(PROCESSED / "matches.csv", index=False)
club_map.to_csv(PROCESSED / "club_map.csv", index=False)

coverage = matches.loc[matches["crowd_status"].ne("not_studied"), "espn_id"].notna().mean()
display(pd.Series({
    "partidas": len(matches),
    "disputadas_pos_pandemia": int((matches["crowd_status"] != "not_studied").sum()),
    "cobertura_da_juncao": round(coverage, 3),
    "regulares": int(matches["crowd_status"].eq("regular").sum()),
    "sem_publico": int(matches["crowd_status"].eq("missing").sum()),
    "com_restricao": int(matches["crowd_status"].eq("restricted").sum()),
}, name="valor").to_frame())
assert matches["game_id"].is_unique, "game_id duplicado"
assert coverage > 0.95, f"Apenas {coverage:.1%} das partidas depois da pandemia foram unidas à ESPN"
assert matches.loc[matches["season"] < 2022, "attendance"].isna().all(), "Há público registrado antes de 2022"

,valor
partidas,2660.0
disputadas_pos_pandemia,1880.0
cobertura_da_juncao,1.0
regulares,1675.0
sem_publico,203.0
com_restricao,2.0
